In [11]:
!rm -rf NN-Project1/

In [12]:
!git clone https://www.github.com/yousefkoriem/NN-Project1.git

Cloning into 'NN-Project1'...
remote: Enumerating objects: 271, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 271 (delta 2), reused 24 (delta 2), pack-reused 247 (from 2)
Receiving objects: 100% (271/271), 221.42 MiB | 30.69 MiB/s, done.
Resolving deltas: 100% (62/62), done.


In [13]:
import os
os.chdir("/content/NN-Project1")

In [14]:
import keras
import tensorflow as tf
import numpy as np
import pandas as pd
from keras import layers
from keras.utils import text_dataset_from_directory
import spacy
import re
import pickle
from keras import layers, models, metrics, optimizers,callbacks

In [15]:
model = models.load_model("models/architecture/untrained_imdb_model.keras")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 16 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [16]:
checkpoint = callbacks.ModelCheckpoint(
    filepath="models/architecture/best_imdb_model.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [17]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [18]:
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [19]:
# 1. Load the raw datasets
train_ds = tf.data.Dataset.load("data/processed/train_ds")
val_ds = tf.data.Dataset.load("data/processed/val_ds")
test_ds = tf.data.Dataset.load("data/processed/test_ds")

# 2. Define the label shape fix
def vectorize_text(text, label):
    return text, tf.expand_dims(label, -1)

# 3. Map the fix to the data
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)

# 4. Apply cache and prefetch exactly ONCE at the very end
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [20]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/100
623/625 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6708 - f1_score: 0.7075 - loss: 0.5574
Epoch 1: val_loss improved from None to 0.30065, saving model to models/architecture/best_imdb_model.keras

Epoch 1: finished saving model to models/architecture/best_imdb_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - accuracy: 0.7847 - f1_score: 0.7923 - loss: 0.4287 - val_accuracy: 0.8720 - val_f1_score: 0.8726 - val_loss: 0.3006 - learning_rate: 0.0010
Epoch 2/100
621/625 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9069 - f1_score: 0.9078 - loss: 0.2359
Epoch 2: val_loss did not improve from 0.30065
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9348 - f1_score: 0.9346 - loss: 0.1724 - val_accuracy: 0.8728 - val_f1_score: 0.8756 - val_loss: 0.3391 - learning_rate: 0.0010
Epoch 3/100
620/625 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9740 - f1_score: 0.9742 - loss: 0.0785
Epoch 3: val_loss did not improve from 0.30065
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9m

In [ ]:
model.summary()

In [ ]:
!mkdir results/tables

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv("results/tables/history.csv", index=False)

In [ ]:
score = model.evaluate(test_ds)
score_df = pd.DataFrame([score], columns=["loss", "accuracy", "f1_score"])

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8568 - f1_score: 0.8491 - loss: 0.3266


In [ ]:
!mkdir results/models

In [ ]:
model.save("results/models/final_imdb_model.keras")